# 🔄 Avaliação Jornada a Jornada (Temporal Rollout)
Este notebook junta as tuas features avançadas (`Modelacao_RandomForest.py`) com o novo sistema de avaliação sequencial da Primeira Liga.

In [ ]:
!pip install shap --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import shap
shap.initjs()

## 1. Carregamento dos Dados Avançados
Carregamos o dataset de features avançadas e estruturamos a numeração temporal da época.

In [ ]:
try:
    # Ajusta o caminho se o Notebook foi movido
    df = pd.read_csv(r"..\Datasets\dataset_features_avancadas.csv", low_memory=False)
    print("Dataset carregado com sucesso!")
except FileNotFoundError:
    try:
        df = pd.read_csv(r"d:\Diogo\Ambiente de Trabalho\PROJETO\Datasets\dataset_features_avancadas.csv", low_memory=False)
        print("Dataset carregado pelo caminho absoluto com sucesso!")
    except:
        print("Erro: Ficheiro não encontrado.")

if "Data" in df.columns:
    df["Data"] = pd.to_datetime(df["Data"], dayfirst=True, errors="coerce")
    df = df.dropna(subset=["Data"]).sort_values("Data").reset_index(drop=True)
    
    # Criar coluna Epoca se não existir
    if "Epoca" not in df.columns:
        def get_season(date):
            return f"{date.year}-{date.year+1}" if date.month >= 7 else f"{date.year-1}-{date.year}"
        df["Epoca"] = df["Data"].apply(get_season)
    
    # Criar Jornada se não existir
    if "Jornada" not in df.columns:
        def compute_jornadas(season_df):
            season_df = season_df.copy()
            team_games = {}
            jornadas = []
            for idx, row in season_df.iterrows():
                home, away = row["Equipa_Casa"], row["Equipa_Visitante"]
                hg, ag = team_games.get(home, 0), team_games.get(away, 0)
                matchday = max(hg, ag) + 1
                jornadas.append(matchday)
                team_games[home] = hg + 1
                team_games[away] = ag + 1
            season_df["Jornada"] = jornadas
            return season_df
        df = df.groupby("Epoca", group_keys=False).apply(compute_jornadas).sort_values("Data").reset_index(drop=True)
        print("Coluna Jornada gerada a partir das Datas.")

## 2. Definição de Features e Criação do Target

In [ ]:
features = [
    "Home_hist_Pontos", "Home_hist_GolosMarcados", "Home_hist_GolosSofridos",
    "Home_hist_DiferençaDeGolos", "Home_hist_Vitórias", "Home_hist_Derrotas", "Home_hist_Empates",
    "Away_hist_Pontos", "Away_hist_GolosMarcados", "Away_hist_GolosSofridos",
    "Away_hist_DiferençaDeGolos", "Away_hist_Vitórias", "Away_hist_Derrotas", "Away_hist_Empates",
    
    "Casa_Form_Pts5", "Casa_Form_GM5", "Casa_Form_GS5",
    "Visitante_Form_Pts5", "Visitante_Form_GM5", "Visitante_Form_GS5",
    "Casa_Form_Empates5", "Visitante_Form_Empates5",
    
    "Home_hist_GolosEsperados", "Home_hist_GolosEsperadosSofridos",
    "Away_hist_GolosEsperados", "Away_hist_GolosEsperadosSofridos",
    "Home_hist_PosseDeBola", "Away_hist_PosseDeBola",
    "Home_hist_PassesProgressivos", "Away_hist_PassesProgressivos",
    "Home_hist_JogosSemSofrerGolos", "Away_hist_JogosSemSofrerGolos",
    
    "Casa_Elo_PreJogo", "Visitante_Elo_PreJogo", "Diff_Elo",
    "Casa_ExpectedGolos", "Visitante_ExpectedGolos", "Prob_Empate_Poisson",
    "Casa_BTTS_Rate_5J", "Visitante_BTTS_Rate_5J",
    "Casa_CleanSheet_Rate_5J", "Visitante_CleanSheet_Rate_5J",
    "Casa_Rolling5_Remates", "Visitante_Rolling5_Remates",
    "Casa_Rolling5_RematesAlvo", "Visitante_Rolling5_RematesAlvo",
    "Casa_Rolling5_Cantos", "Visitante_Rolling5_Cantos",
    
    "Jornada"
]

df = df.dropna(subset=features)

le = LabelEncoder()
df["Resultado_Final_Encoded"] = le.fit_transform(df["Resultado_Final"])
target = "Resultado_Final_Encoded"
print(f"Classes Mapeadas: {list(zip(le.classes_, le.transform(le.classes_)))}")

# ==================================================================
# TREINO: Todas as Épocas Históricas (< 2023-2024)
# TESTE: Apenas a Época Alvo para Validação do Título (2023-2024)
# ==================================================================
train_df = df[df["Epoca"] != "2023-2024"].copy()
test_df = df[df["Epoca"] == "2023-2024"].copy()

print(f"\nTreino: {len(train_df)} jogos (Dados Históricos) | Teste: {len(test_df)} jogos (Época 2023-2024)")

## 3. Estrutura Direcionada do Temporal Rollout

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

def executar_temporal_rollout(train_df, test_df, model, features, target):
    print("A treinar base histórico antes de arrancar a Época Teste...")
    model.fit(train_df[features], train_df[target])
    
    # Pegamos nas jornadas únicas ordenadas da secção de Teste
    lista_jornadas = sorted(test_df["Jornada"].unique())
    
    equipas_da_epoca = test_df["Equipa_Casa"].unique()
    tabela_classificativa = {equipa: 0 for equipa in equipas_da_epoca}
    
    analise_campeao_por_jornada = {}
    shap_data = {}
    todas_as_previsoes = []
    
    for jornada in lista_jornadas:
        jogos_desta_jornada = test_df[test_df["Jornada"] == jornada]
        jogos_futuros = test_df[test_df["Jornada"] > jornada]
        
        if jogos_desta_jornada.empty:
            continue
            
        # GUARDAR SHAP VALUES (IMPORTÂNCIA CIENTÍFICA LOCAL) NA JORNADA 1 E ULTIMA JORNADA
        if jornada == lista_jornadas[0] or jornada == lista_jornadas[-1]:
            explainer = shap.TreeExplainer(model)
            shap_vals = explainer.shap_values(jogos_desta_jornada[features])
            shap_data[jornada] = {
                "valores_shap": shap_vals,
                "features_jogos": jogos_desta_jornada[features].copy()
            }

        # PREVISÃO DA JORNADA ATUAL
        preds_jornada = model.predict(jogos_desta_jornada[features])
        
        for iter_idx, (idx, row) in enumerate(jogos_desta_jornada.iterrows()):
            todas_as_previsoes.append({
                "Jornada": jornada,
                "Equipa_Casa": row["Equipa_Casa"],
                "Equipa_Visitante": row["Equipa_Visitante"],
                "Previsao": preds_jornada[iter_idx],
                "Real": row[target]
            })
            
        # SIMULAR QUEM É O CAMPEÃO DA ÉPOCA NO MOMENTO (Prevendo a jornada atual e restos dos jogos da época)
        tabela_simulada = tabela_classificativa.copy()
        
        for iter_idx, (idx, row) in enumerate(jogos_desta_jornada.iterrows()):
            prev = preds_jornada[iter_idx]
            res_str = le.classes_[prev]
            if res_str == "H":   tabela_simulada[row["Equipa_Casa"]] += 3
            elif res_str == "D": 
                tabela_simulada[row["Equipa_Casa"]] += 1
                tabela_simulada[row["Equipa_Visitante"]] += 1
            elif res_str == "A": tabela_simulada[row["Equipa_Visitante"]] += 3
                
        if not jogos_futuros.empty:
            preds_futuro = model.predict(jogos_futuros[features])
            for iter_idx, (idx, row) in enumerate(jogos_futuros.iterrows()):
                prev_fut = preds_futuro[iter_idx]
                res_str = le.classes_[prev_fut]
                if res_str == "H":   tabela_simulada[row["Equipa_Casa"]] += 3
                elif res_str == "D": 
                    tabela_simulada[row["Equipa_Casa"]] += 1
                    tabela_simulada[row["Equipa_Visitante"]] += 1
                elif res_str == "A": tabela_simulada[row["Equipa_Visitante"]] += 3
                    
        try:
            classificacao = sorted(tabela_simulada.items(), key=lambda x: x[1], reverse=True)
            campeao_simulado, pontos_campeao = classificacao[0][0], classificacao[0][1]
            analise_campeao_por_jornada[jornada] = (campeao_simulado, pontos_campeao)
        except:
            pass
            
        # ATUALIZAR REALIDADE DO CAMPEONATO DA ÉPOCA
        for idx, row in jogos_desta_jornada.iterrows():
            res_real = le.classes_[row[target]]
            if res_real == "H":   tabela_classificativa[row["Equipa_Casa"]] += 3
            elif res_real == "D": 
                tabela_classificativa[row["Equipa_Casa"]] += 1
                tabela_classificativa[row["Equipa_Visitante"]] += 1
            elif res_real == "A": tabela_classificativa[row["Equipa_Visitante"]] += 3
                
        # RE-FIT COM OS NOVOS DADOS NA ÚLTIMA JORNADA
        train_df = pd.concat([train_df, jogos_desta_jornada], ignore_index=True)
        model.fit(train_df[features], train_df[target])
        
    return pd.DataFrame(todas_as_previsoes), analise_campeao_por_jornada, shap_data

## 4. Resultados Finais: Época 2023-2024

In [ ]:
resultados_df, historico_campeoes, dados_shap = executar_temporal_rollout(train_df, test_df, rf_model, features, target)

resultados_df["Acertou"] = resultados_df["Previsao"] == resultados_df["Real"]
acuracia = resultados_df["Acertou"].mean()
print(f"\n>> Accuracy Final (Temporal Rollout, Época 2023/24): {acuracia:.3f}")

print("\n--- Relatório Final de Classificação ---")
print(classification_report(resultados_df["Real"], resultados_df["Previsao"], target_names=le.classes_))

cm = confusion_matrix(resultados_df["Real"], resultados_df["Previsao"])
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Matriz de Confusão - Temporal Rollout")
plt.ylabel("Real")
plt.xlabel("Previsão")
plt.show()

print("\nHistórico do Campeão Simulado (Final de cada Jornada):")
for j, (camp, pts) in list(historico_campeoes.items()):
    print(f" - Pós-Jornada {j:02d}: {camp} com {pts} pts")

## 5. Análise SHAP: Impacto Causal nos 9 Jogos da Primeira vs Última Jornada

In [ ]:
if dados_shap:
    jornada_inicial = min(dados_shap.keys())
    jornada_final = max(dados_shap.keys())
    
    print(f"\n{'='*70}")
    print(f"--- SHAP Local (Causalidade): Jornada {jornada_inicial} ---")
    print(f"{'='*70}")
    
    shap.summary_plot(
        dados_shap[jornada_inicial]["valores_shap"], 
        dados_shap[jornada_inicial]["features_jogos"], 
        plot_type="bar", 
        class_names=list(le.classes_), 
        show=False, 
        max_display=12
    )
    plt.title(f"Variáveis que Decidiram os Exatos 9 Duelos da Jornada {jornada_inicial}", fontsize=12)
    plt.show()

    print(f"\n{'='*70}")
    print(f"--- SHAP Local (Causalidade): Jornada {jornada_final} ---")
    print(f"{'='*70}")
    
    shap.summary_plot(
        dados_shap[jornada_final]["valores_shap"], 
        dados_shap[jornada_final]["features_jogos"], 
        plot_type="bar", 
        class_names=list(le.classes_), 
        show=False, 
        max_display=12
    )
    plt.title(f"Variáveis que Decidiram os Exatos 9 Duelos da Jornada {jornada_final}", fontsize=12)
    plt.show()